[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jujumelona/material-candidate-explorer/blob/main/MATERIAL_CANDIDATE_DISCOVERY_T4.ipynb)

# Goal

Run a real, evidence-preserving material-candidate search on a Google Colab T4. Four independent MatterGen profiles generate exactly the requested 8-32 candidates. MatterSim and CHGNet remain separate experts; the notebook never averages their energies or treats a generator target as a measured thermodynamic result. Optional API keys and MCP tokens are requested with hidden prompts in Setup and are never written to the result archive.

In [ ]:
# @title 0. Search settings
DISCOVERY_PROMPT = "Find low-energy Li-O crystal candidates while preserving structural diversity." # @param {type:"string"}
CHEMICAL_SYSTEM = "Li-O" # @param {type:"string"}
TOTAL_CANDIDATES = 16 # @param {type:"integer", min:8, max:32}
DFT_TOP_K = 3 # @param {type:"integer", min:1, max:5}
BASE_SEED = 20260716 # @param {type:"integer"}
MINIMUM_DISTANCE_A = 0.70 # @param {type:"number"}
RELAX_STEPS = 150 # @param {type:"integer", min:1, max:2000}
RELAX_FMAX_EV_A = 0.05 # @param {type:"number"}
INSTALL_OR_REUSE_MODELS = True # @param {type:"boolean"}
DOWNLOAD_RESULTS_ZIP = True # @param {type:"boolean"}
ENABLE_STAGE_EVIDENCE = True # @param {type:"boolean"}
VALIDATION_EVIDENCE_MAX_RESULTS = 8 # @param {type:"integer", min:1, max:50}
VALIDATION_EVIDENCE_MAX_BRANCHES = 12 # @param {type:"integer", min:1, max:50}
VALIDATION_EVIDENCE_FROM_DATE = "" # @param {type:"string"}
LITERATURE_CONTACT_EMAIL = "" # @param {type:"string"}
RAG_MODEL_API_URL = "" # @param {type:"string"}
RAG_MODEL_NAME = "" # @param {type:"string"}
MATERIAL_RAG_MCP_URL = "" # @param {type:"string"}
MATERIAL_RAG_MCP_TOOL = "" # @param {type:"string"}

import re
from datetime import date

if not DISCOVERY_PROMPT.strip():
    raise ValueError("DISCOVERY_PROMPT is required.")
if not 8 <= TOTAL_CANDIDATES <= 32:
    raise ValueError("TOTAL_CANDIDATES must be between 8 and 32.")
if not 1 <= DFT_TOP_K <= 5:
    raise ValueError("DFT_TOP_K must be between 1 and 5.")
if not 1 <= RELAX_STEPS <= 2000:
    raise ValueError("RELAX_STEPS must be between 1 and 2000.")
if MINIMUM_DISTANCE_A <= 0 or RELAX_FMAX_EV_A <= 0:
    raise ValueError("Geometry and relaxation thresholds must be positive.")
if not 1 <= VALIDATION_EVIDENCE_MAX_RESULTS <= 50:
    raise ValueError("VALIDATION_EVIDENCE_MAX_RESULTS must be between 1 and 50.")
if not 1 <= VALIDATION_EVIDENCE_MAX_BRANCHES <= 50:
    raise ValueError("VALIDATION_EVIDENCE_MAX_BRANCHES must be between 1 and 50.")
if VALIDATION_EVIDENCE_FROM_DATE.strip():
    date.fromisoformat(VALIDATION_EVIDENCE_FROM_DATE.strip())
if bool(RAG_MODEL_API_URL.strip()) != bool(RAG_MODEL_NAME.strip()):
    raise ValueError("RAG_MODEL_API_URL and RAG_MODEL_NAME must be set together.")
if bool(MATERIAL_RAG_MCP_URL.strip()) != bool(MATERIAL_RAG_MCP_TOOL.strip()):
    raise ValueError("MATERIAL_RAG_MCP_URL and MATERIAL_RAG_MCP_TOOL must be set together.")

PERIODIC_TABLE = set(
    "H He Li Be B C N O F Ne Na Mg Al Si P S Cl Ar K Ca Sc Ti V Cr Mn Fe Co Ni "
    "Cu Zn Ga Ge As Se Br Kr Rb Sr Y Zr Nb Mo Tc Ru Rh Pd Ag Cd In Sn Sb Te I "
    "Xe Cs Ba La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu Hf Ta W Re Os Ir Pt "
    "Au Hg Tl Pb Bi Po At Rn Fr Ra Ac Th Pa U Np Pu Am Cm Bk Cf Es Fm Md No Lr "
    "Rf Db Sg Bh Hs Mt Ds Rg Cn Nh Fl Mc Lv Ts Og".split()
)
symbols = [item for item in re.split(r"[-,\s]+", CHEMICAL_SYSTEM.strip()) if item]
if not 1 <= len(symbols) <= 16 or any(item not in PERIODIC_TABLE for item in symbols):
    raise ValueError("Use 1-16 element symbols, for example Li-O.")
chemical_system = "-".join(dict.fromkeys(symbols))

PROFILE_TEMPLATES = [
    {"profile_id": "tight", "guidance_alpha": 0.25, "target_hull": 0.00},
    {"profile_id": "balanced", "guidance_alpha": 0.45, "target_hull": 0.03},
    {"profile_id": "broad", "guidance_alpha": 0.65, "target_hull": 0.06},
    {"profile_id": "explore", "guidance_alpha": 0.85, "target_hull": 0.10},
]
base_count, remainder = divmod(TOTAL_CANDIDATES, len(PROFILE_TEMPLATES))
PROFILES = [
    {**profile, "candidate_count": base_count + (index < remainder)}
    for index, profile in enumerate(PROFILE_TEMPLATES)
]
assert sum(item["candidate_count"] for item in PROFILES) == TOTAL_CANDIDATES
assert all(item["candidate_count"] >= 2 for item in PROFILES)
print(PROFILES)


## Setup

Select Runtime > Change runtime type > T4 GPU, then run the next cell. Model environments stay isolated behind local HTTP sidecars. Materials Project and enabled evidence-service credentials are entered with hidden input and are never written to artifacts.

In [ ]:
# @title 1. Install the public project and start isolated sidecars
from __future__ import annotations

import getpass
import json
import os
import shlex
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPOSITORY_URL = "https://github.com/jujumelona/material-candidate-explorer.git"
REPOSITORY_DIR = Path("/content/material-candidate-explorer")
RUN_ID = "material-t4-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ROOT = Path("/content") / RUN_ID
ARTIFACT_ROOT = RUN_ROOT / "artifacts"
AUDIT_CIF_ROOT = RUN_ROOT / "candidate-cifs-raw-audit"
UNIQUE_CIF_ROOT = RUN_ROOT / "crystallographically-unique-cifs"
RUN_ROOT.mkdir(parents=True, exist_ok=False)
ARTIFACT_ROOT.mkdir()
AUDIT_CIF_ROOT.mkdir()
UNIQUE_CIF_ROOT.mkdir()

def run_checked(command: list[str], *, cwd: Path | None = None, capture: bool = False):
    print("$", " ".join(shlex.quote(item) for item in command))
    return subprocess.run(
        command, cwd=cwd, check=True, text=True,
        capture_output=capture,
    )

if not (3, 11) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("The pinned materials extra requires Python 3.11-3.12.")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select a Colab T4 GPU runtime before continuing.")

if not (REPOSITORY_DIR / ".git").is_dir():
    run_checked(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_DIR)])
else:
    dirty = run_checked(
        ["git", "-C", str(REPOSITORY_DIR), "status", "--porcelain", "--untracked-files=no"],
        capture=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError("The public clone has tracked edits; restart Colab before updating.")
    run_checked(["git", "-C", str(REPOSITORY_DIR), "fetch", "--depth", "1", "origin", "main"])
    run_checked(["git", "-C", str(REPOSITORY_DIR), "checkout", "main"])
    run_checked(["git", "-C", str(REPOSITORY_DIR), "merge", "--ff-only", "origin/main"])

run_checked([
    sys.executable, "-m", "pip", "install", "-e", ".[materials]",
], cwd=REPOSITORY_DIR)

os.environ.update({
    "MATTERGEN_PRETRAINED_NAME": "chemical_system_energy_above_hull",
    "MATTERGEN_DEVICE": "cuda",
    "MATTERSIM_DEVICE": "cpu",
    "CHGNET_DEVICE": "cpu",
    "MATTERSIM_WEIGHT_REVISION": "managed-unattested:mattersim-1.2.5-upstream-default",
    "CHGNET_WEIGHT_REVISION": "managed-unattested:chgnet-0.3.0-builtin",
})
if INSTALL_OR_REUSE_MODELS:
    run_checked([
        "bash", "bootstrap.sh", "--profile", "materials-open",
        "--accelerator", "cuda", "--include-weights",
    ], cwd=REPOSITORY_DIR)
    run_checked([
        "bash", "start-sidecars.sh", "--profile", "materials-open",
        "--component", "mattergen", "--component", "mattersim",
        "--component", "chgnet", "--ready-timeout-seconds", "900",
    ], cwd=REPOSITORY_DIR)

environment_file = REPOSITORY_DIR / ".discovery" / "sidecars.env.sh"
if not environment_file.is_file():
    raise RuntimeError("Sidecar environment was not created.")
raw_environment = run_checked([
    "bash", "-lc",
    "set -a; . " + shlex.quote(str(environment_file)) + "; env -0",
], capture=True).stdout.encode()
for entry in raw_environment.split(b"\0"):
    if b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[key.decode()] = value.decode()

mp_key = getpass.getpass(
    "Materials Project API key (optional, hidden; press Enter to skip): "
).strip()
if mp_key:
    os.environ["MP_API_KEY"] = mp_key
del mp_key

stage_environment = {
    "VALIDATION_EVIDENCE_ENABLED": "1" if ENABLE_STAGE_EVIDENCE else "0",
    "VALIDATION_EVIDENCE_MAX_RESULTS": str(VALIDATION_EVIDENCE_MAX_RESULTS),
    "VALIDATION_EVIDENCE_MAX_BRANCHES": str(VALIDATION_EVIDENCE_MAX_BRANCHES),
    "VALIDATION_EVIDENCE_FROM_DATE": VALIDATION_EVIDENCE_FROM_DATE.strip(),
    "LITERATURE_CONTACT_EMAIL": LITERATURE_CONTACT_EMAIL.strip(),
    "RAG_MODEL_API_URL": RAG_MODEL_API_URL.strip(),
    "RAG_MODEL_NAME": RAG_MODEL_NAME.strip(),
    "MATERIAL_RAG_MCP_URL": MATERIAL_RAG_MCP_URL.strip(),
    "MATERIAL_RAG_MCP_TOOL": MATERIAL_RAG_MCP_TOOL.strip(),
}
for key, value in stage_environment.items():
    if value:
        os.environ[key] = value

if ENABLE_STAGE_EVIDENCE:
    hidden_prompts = [
        (
            "OPENALEX_API_KEY",
            "OpenAlex API key (hidden; press Enter to skip the OpenAlex source): ",
            True,
        ),
        (
            "RAG_MODEL_API_KEY",
            "RAG model API key (optional, hidden; press Enter if not required): ",
            bool(RAG_MODEL_API_URL.strip()),
        ),
        (
            "MATERIAL_RAG_MCP_TOKEN",
            "Material MCP bearer token (optional, hidden; press Enter if not required): ",
            bool(MATERIAL_RAG_MCP_URL.strip()),
        ),
    ]
    for variable, prompt, required_by_configuration in hidden_prompts:
        if not required_by_configuration:
            continue
        secret_value = getpass.getpass(prompt).strip()
        if secret_value:
            os.environ[variable] = secret_value
        del secret_value
del stage_environment

import requests

health_requirements = {
    "mattergen": ("MATTERGEN_API_URL", {"generate"}),
    "mattersim": ("MATTERSIM_API_URL", {"features", "relax"}),
    "chgnet": ("CHGNET_API_URL", {"features", "relax"}),
}
health_rows = []
for component, (variable, required_operations) in health_requirements.items():
    endpoint = os.environ.get(variable, "").rstrip("/")
    if not endpoint:
        raise RuntimeError(variable + " is missing.")
    response = requests.get(endpoint + "/health", timeout=30)
    response.raise_for_status()
    payload = response.json()
    operations = set(payload.get("operations", []))
    if not required_operations.issubset(operations):
        raise RuntimeError(
            component + " is missing operations: " +
            ", ".join(sorted(required_operations - operations))
        )
    health_rows.append({
        "component": component,
        "status": payload.get("status"),
        "operations": sorted(operations),
        "weight_revision": payload.get("weight_revision"),
    })

print(json.dumps(health_rows, indent=2))
print("Run root:", RUN_ROOT)


## Steps

1. Allocate the total batch over tight, balanced, broad, and explore profiles. 2. Run one strict fusion-iterate branch per profile. 3. Parse the authoritative Candidate CIF only. 4. Deduplicate exact files and then crystals across all profiles. 5. Evaluate and relax every safe unique crystal with both MLIPs. 6. Rank only within reduced composition, assess staged novelty, and package 1-5 DFT handoffs.

In [ ]:
# @title 2. Build four profile-bound goals and run configurations
from discovery_os.fusion_schemas import (
    GenerationControls,
    WorkspaceMode,
    WorkspaceRunConfig,
)
from discovery_os.hashing import candidate_content_hash, stable_hash
from discovery_os.integration_manifest import load_integration_manifest
from discovery_os.materials_screening import GenerationConditions
from discovery_os.validation_evidence import (
    ValidationEvidenceRequest,
    ValidationEvidenceStage,
    build_validation_evidence_router_from_environment,
    fusion_decision_contexts_from_stage_evidence,
)
from discovery_os.schemas import (
    Candidate,
    CandidateRef,
    CandidateRepresentation,
    CandidateType,
    DiscoveryDomain,
    DiscoveryGoal,
    ObjectiveDirection,
    PropertyObjective,
    RepresentationKind,
)

def ref_key(reference: CandidateRef) -> tuple[str, int, str]:
    return (
        reference.candidate_id,
        reference.version,
        reference.content_hash,
    )

def seed_cif(elements: list[str]) -> str:
    rows = []
    for index, element in enumerate(elements):
        x = (index % 4) / 4.0
        y = ((index // 4) % 4) / 4.0
        z = ((index // 16) % 4) / 4.0
        rows.append(f"{element}{index + 1} {element} {x:.6f} {y:.6f} {z:.6f} 1.0")
    return "\n".join([
        "data_material_search_seed",
        "_symmetry_space_group_name_H-M 'P 1'",
        "_symmetry_Int_Tables_number 1",
        "_cell_length_a 8.000",
        "_cell_length_b 8.000",
        "_cell_length_c 8.000",
        "_cell_angle_alpha 90.000",
        "_cell_angle_beta 90.000",
        "_cell_angle_gamma 90.000",
        "loop_",
        "_atom_site_label",
        "_atom_site_type_symbol",
        "_atom_site_fract_x",
        "_atom_site_fract_y",
        "_atom_site_fract_z",
        "_atom_site_occupancy",
        *rows,
        "",
    ])

draft_parent = Candidate(
    candidate_id="colab-material-lineage-seed",
    candidate_type=CandidateType.CRYSTAL,
    domain=DiscoveryDomain.GENERAL_MATERIALS,
    name=chemical_system + " lineage seed",
    representations=[CandidateRepresentation(
        kind=RepresentationKind.CIF,
        value=seed_cif(symbols),
        media_type="chemical/x-cif",
        format_version="CIF 1.1",
        canonical=False,
    )],
    attributes={"chemical_system": chemical_system, "seed_only": True},
    provenance={"role": "lineage_seed_not_a_stability_claim"},
)
parent_data = draft_parent.model_dump(mode="python")
parent_data["candidate_ref"] = CandidateRef(
    candidate_id=draft_parent.candidate_id,
    version=1,
    content_hash=candidate_content_hash(draft_parent),
)
parent = Candidate.model_validate(parent_data)
PARENT_PATH = RUN_ROOT / "parent.json"
PARENT_PATH.write_text(parent.model_dump_json(indent=2), encoding="utf-8")

manifest = load_integration_manifest()
mattergen_component = next(
    item for item in manifest.components if item.component_id == "mattergen"
)
mattergen_weight_revision = os.environ["MATTERGEN_WEIGHT_REVISION"].strip()

profile_inputs = []
for index, profile in enumerate(PROFILES):
    conditions = GenerationConditions(
        profile_id=profile["profile_id"],
        guidance_alpha=profile["guidance_alpha"],
        target_energy_above_hull_eV_atom=profile["target_hull"],
    )
    goal = DiscoveryGoal(
        goal_id="material-goal-" + profile["profile_id"],
        domain=DiscoveryDomain.GENERAL_MATERIALS,
        title=chemical_system + " " + profile["profile_id"] + " crystal branch",
        scientific_question=DISCOVERY_PROMPT.strip(),
        objectives=[
            PropertyObjective(
                property_name="energy_per_atom",
                direction=ObjectiveDirection.MINIMIZE,
                unit="eV/atom",
                weight=1.0,
                required=True,
                rationale="Screen separately with MatterSim and CHGNet.",
            ),
            PropertyObjective(
                property_name="max_force",
                direction=ObjectiveDirection.MINIMIZE,
                unit="eV/angstrom",
                weight=0.5,
                required=True,
                rationale="Reject unrelaxed or collapsing structures.",
            ),
            PropertyObjective(
                property_name="energy_above_hull",
                direction=ObjectiveDirection.TARGET,
                unit="eV/atom",
                target_value=profile["target_hull"],
                weight=0.0,
                required=False,
                rationale=(
                    "MatterGen generation condition only; no hull value is measured "
                    "or inferred by this objective."
                ),
            ),
        ],
        validation_profile_id="dual-mlip-periodic-screening-v1",
        candidate_types=[CandidateType.CRYSTAL],
        assumptions=[
            "MatterGen targets are generation conditions, not thermodynamic validation.",
            "Cross-stoichiometry absolute MLIP energies are not comparable.",
        ],
        max_cycles=1,
    )
    controls = GenerationControls(
        alpha=profile["guidance_alpha"],
        temperature=1.0,
        mutation_strength=0.0,
        diversity_strength=profile["guidance_alpha"],
        schedule_step=index,
        decision_reason=profile["profile_id"] + " independent profile branch",
    )
    config = WorkspaceRunConfig(
        workspace_mode=WorkspaceMode.ON,
        seed=BASE_SEED + index * 1000,
        generator_seed=BASE_SEED + index * 1000,
        goal_hash=stable_hash(goal),
        parent_candidate_ref=parent.candidate_ref,
        pair_key="colab-material-" + profile["profile_id"],
        cohort_index=index,
        generator_id="mattergen",
        generator_version=mattergen_component.install.version,
        generator_code_revision=mattergen_component.source.revision,
        generator_weight_revision=mattergen_weight_revision,
        generator_parameters_hash=stable_hash({
            "pretrained_name": os.environ["MATTERGEN_PRETRAINED_NAME"],
            "chemical_system": chemical_system,
            "generation_conditions": conditions,
            "profile_candidate_count": profile["candidate_count"],
        }),
        decoder_config_hash=stable_hash({"decoder": "mattergen-upstream-cif"}),
        postprocessing_hash=stable_hash({
            "authoritative_representation": "candidate-cif-only",
            "cross_profile_structure_match": "deferred-to-notebook",
        }),
        resource_budget_hash=stable_hash({"runtime": "colab-t4", "branch": index}),
        evaluator_panel_hash=stable_hash({"experts": ["mattersim", "chgnet"]}),
        candidate_count=profile["candidate_count"],
        generation_controls=controls,
    )
    profile_dir = RUN_ROOT / "profiles" / profile["profile_id"]
    profile_dir.mkdir(parents=True)
    goal_path = profile_dir / "goal.json"
    config_path = profile_dir / "run-config.json"
    goal_path.write_text(goal.model_dump_json(indent=2), encoding="utf-8")
    config_path.write_text(config.model_dump_json(indent=2), encoding="utf-8")
    profile_inputs.append({
        "profile": profile,
        "conditions": conditions,
        "goal": goal,
        "config": config,
        "goal_path": goal_path,
        "config_path": config_path,
    })

stage_evidence_router = build_validation_evidence_router_from_environment(
    artifact_root=RUN_ROOT,
    enabled=ENABLE_STAGE_EVIDENCE,
)
stage_evidence_runs = {}
generation_evidence_run = stage_evidence_router.run(
    ValidationEvidenceRequest(
        stage=ValidationEvidenceStage.GENERATION_PRIOR,
        chemical_system=chemical_system,
        observations={
            "user_prompt": DISCOVERY_PROMPT.strip(),
            "profiles": [
                item["conditions"].model_dump(mode="json")
                for item in profile_inputs
            ],
            "requested_candidate_count": TOTAL_CANDIDATES,
            "generation_model": "MatterGen",
            "numeric_property_scores_created": False,
        },
        focus=(
            "Retrieve experimentally reported phases, failed or negative synthesis "
            "evidence, stability constraints, and supported MatterGen conditions."
        ),
    ),
    goal=profile_inputs[0]["goal"],
)
stage_evidence_runs["generation_prior"] = generation_evidence_run
profile_branches = ["stability", "pareto", "target_property", "novelty"]
profile_contexts = fusion_decision_contexts_from_stage_evidence(
    generation_evidence_run,
    controls=[
        (item["profile"]["guidance_alpha"], profile_branches[index])
        for index, item in enumerate(profile_inputs)
    ],
)
for item, decision_context in zip(profile_inputs, profile_contexts):
    context_path = (
        RUN_ROOT / "profiles" / item["profile"]["profile_id"]
        / "decision-context.json"
    )
    context_path.write_text(
        decision_context.model_dump_json(indent=2), encoding="utf-8"
    )
    item["decision_context"] = decision_context
    item["decision_context_path"] = context_path

print(json.dumps([
    {
        **item["conditions"].model_dump(mode="json"),
        "candidate_count": item["config"].candidate_count,
        "goal_hash": item["config"].goal_hash,
    }
    for item in profile_inputs
], indent=2))


In [ ]:
# @title 3. Run one real fusion iteration for every profile
from discovery_os.fusion_schemas import FusionBatchIterationReport

def raw_generation_count(raw_report: dict) -> int:
    generation = raw_report["generation"]
    if generation.get("candidates"):
        return len(generation["candidates"])
    return 1 if generation.get("candidate") is not None else 0

generated_rows = []
profile_reports = {}
requested_count = sum(item["config"].candidate_count for item in profile_inputs)
returned_candidate_count = 0
raw_model_structure_count = 0
parsed_model_structure_count = 0
profile_generation_funnels = {}
profile_generation_funnel_hashes = {}
raw_exact_file_hashes = set()
cli_environment = os.environ.copy()
for secret_name in (
    "MP_API_KEY",
    "OPENALEX_API_KEY",
    "RAG_MODEL_API_KEY",
    "MATERIAL_RAG_MCP_TOKEN",
):
    cli_environment.pop(secret_name, None)

for item in profile_inputs:
    profile_id = item["profile"]["profile_id"]
    command = [
        sys.executable, "-m", "discovery_os.cli", "fusion-iterate",
        "--goal", str(item["goal_path"]),
        "--parent", str(PARENT_PATH),
        "--run-config", str(item["config_path"]),
        "--decision-context", str(item["decision_context_path"]),
        "--generator", "mattergen",
        "--cycle", "0",
        "--expert", "mattersim",
        "--expert", "chgnet",
        "--artifacts", str(ARTIFACT_ROOT),
        "--workspace-id", "colab-material-" + profile_id,
    ]
    print("$", " ".join(shlex.quote(part) for part in command))
    completed = subprocess.run(
        command,
        cwd=REPOSITORY_DIR,
        env=cli_environment,
        check=True,
        text=True,
        capture_output=True,
    )
    raw_report = json.loads(completed.stdout)
    returned_profile_count = raw_generation_count(raw_report)
    returned_candidate_count += returned_profile_count
    report = FusionBatchIterationReport.model_validate_json(
        completed.stdout,
        strict=True,
    )
    expected = item["config"].candidate_count
    if returned_profile_count != expected:
        raise RuntimeError(
            f"{profile_id}: returned {returned_profile_count} unique candidates, expected {expected}."
        )
    candidate_funnels = [
        candidate.attributes.get("generation_funnel")
        for candidate in report.generation.generated_candidates
    ]
    if any(not isinstance(value, dict) for value in candidate_funnels):
        raise RuntimeError(f"{profile_id}: MatterGen generation_funnel is missing.")
    profile_funnel = candidate_funnels[0]
    if any(value != profile_funnel for value in candidate_funnels[1:]):
        raise RuntimeError(f"{profile_id}: candidates disagree on the raw generation funnel.")
    required_funnel_keys = {
        "requested_samples", "raw_model_structures", "parsed_structures",
        "exact_file_unique", "crystallographically_unique", "geometry_valid",
    }
    if not required_funnel_keys.issubset(profile_funnel):
        raise RuntimeError(f"{profile_id}: MatterGen generation_funnel is incomplete.")
    if int(profile_funnel["requested_samples"]) != expected:
        raise RuntimeError(f"{profile_id}: requested sample audit does not match the run config.")
    if int(profile_funnel["crystallographically_unique"]) != expected:
        raise RuntimeError(f"{profile_id}: upstream unique-candidate audit does not match.")
    candidate_hash_payloads = [
        candidate.attributes.get("generation_funnel_hashes")
        for candidate in report.generation.generated_candidates
    ]
    if any(not isinstance(value, dict) for value in candidate_hash_payloads):
        raise RuntimeError(f"{profile_id}: generation_funnel_hashes is missing.")
    profile_hash_payload = candidate_hash_payloads[0]
    if any(value != profile_hash_payload for value in candidate_hash_payloads[1:]):
        raise RuntimeError(f"{profile_id}: candidates disagree on funnel hashes.")
    exact_hashes = profile_hash_payload.get("exact_file_sha256s")
    if (
        not isinstance(exact_hashes, list)
        or len(set(exact_hashes)) != int(profile_funnel["exact_file_unique"])
        or any(
            not isinstance(value, str)
            or re.fullmatch(r"[0-9a-f]{64}", value) is None
            for value in exact_hashes
        )
    ):
        raise RuntimeError(f"{profile_id}: exact-file hash audit is invalid.")
    profile_generation_funnels[profile_id] = dict(profile_funnel)
    profile_generation_funnel_hashes[profile_id] = dict(profile_hash_payload)
    raw_exact_file_hashes.update(exact_hashes)
    raw_model_structure_count += int(profile_funnel["raw_model_structures"])
    parsed_model_structure_count += int(profile_funnel["parsed_structures"])
    if len(report.generation.generated_candidates) != expected:
        raise RuntimeError(f"{profile_id}: strict report candidate count changed.")
    if len(report.after_revisions) != expected:
        raise RuntimeError(f"{profile_id}: every candidate requires a reevaluation report.")
    if len(report.generation.pair_slots) != expected:
        raise RuntimeError(f"{profile_id}: pair-slot provenance is incomplete.")
    report_path = RUN_ROOT / "profiles" / profile_id / "fusion-iteration.json"
    report_path.write_text(report.model_dump_json(indent=2), encoding="utf-8")
    profile_reports[profile_id] = report
    for candidate, after_report, slot in zip(
        report.generation.generated_candidates,
        report.after_revisions,
        report.generation.pair_slots,
        strict=True,
    ):
        generated_rows.append({
            "profile_id": profile_id,
            "profile": item["profile"],
            "conditions": item["conditions"],
            "candidate": candidate,
            "after_report": after_report,
            "pair_slot": slot.model_dump(mode="json"),
        })

if requested_count != TOTAL_CANDIDATES or returned_candidate_count != requested_count:
    raise RuntimeError(
        "The exact requested batch was not returned after upstream replacement sampling."
    )
print({
    "fusion_iterate_calls": len(profile_inputs),
    "requested_samples": requested_count,
    "returned_unique_candidates": returned_candidate_count,
    "raw_model_structures": raw_model_structure_count,
    "parsed_structures": parsed_model_structure_count,
    "global_raw_exact_file_unique": len(raw_exact_file_hashes),
})


In [ ]:
# @title 4. Apply the exact-file and cross-profile crystal-identity funnel
from discovery_os.crystal_identity import (
    canonicalize_crystal_structure,
    exact_file_hash,
    group_crystal_structures,
    parse_crystal_structure,
    validate_crystal_geometry,
)

from discovery_os.novelty import (
    MaterialsProjectStructureLookup,
    StagedNoveltyAssessor,
)

identity_failures = []
parsed_rows = []
for row in generated_rows:
    candidate = row["candidate"]
    cif_representations = [
        representation
        for representation in candidate.representations
        if representation.kind == RepresentationKind.CIF
    ]
    if len(cif_representations) != 1:
        identity_failures.append({
            "candidate_ref": candidate.candidate_ref.model_dump(mode="json"),
            "stage": "parsed",
            "error": "candidate_must_have_exactly_one_authoritative_cif",
        })
        continue
    cif = cif_representations[0]
    try:
        structure = parse_crystal_structure(cif.value, fmt="cif")
    except Exception as exc:
        identity_failures.append({
            "candidate_ref": candidate.candidate_ref.model_dump(mode="json"),
            "stage": "parsed",
            "error": type(exc).__name__ + ": " + str(exc),
        })
        continue
    digest = exact_file_hash(cif.value)
    output_name = (
        row["profile_id"] + "-" + str(row["pair_slot"]["pair_slot"]).zfill(3)
        + "-" + digest[:12] + ".cif"
    )
    output_path = AUDIT_CIF_ROOT / output_name
    output_path.write_text(cif.value, encoding="utf-8")
    parsed_rows.append({
        **row,
        "cif": cif,
        "structure": structure,
        "exact_file_sha256": digest,
        "raw_audit_cif_path": str(output_path.relative_to(RUN_ROOT)),
    })

exact_unique_rows = []
seen_exact_hashes = set()
for row in parsed_rows:
    if row["exact_file_sha256"] in seen_exact_hashes:
        continue
    seen_exact_hashes.add(row["exact_file_sha256"])
    exact_unique_rows.append(row)

canonicalizable_rows = []
for row in exact_unique_rows:
    try:
        canonical = canonicalize_crystal_structure(
            row["structure"],
            minimum_distance_angstrom=1.0e-4,
        )
    except Exception as exc:
        identity_failures.append({
            "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
            "stage": "crystallographically-unique",
            "error": type(exc).__name__ + ": " + str(exc),
        })
        continue
    canonicalizable_rows.append({**row, "canonical": canonical})

grouping = group_crystal_structures(
    tuple(row["canonical"] for row in canonicalizable_rows),
    ltol=0.2,
    stol=0.3,
    angle_tol=5.0,
)
crystal_group_records = []
crystal_unique_rows = []
for group_index, group in enumerate(grouping.groups, 1):
    members = [canonicalizable_rows[index] for index in group.member_indices]
    representative = canonicalizable_rows[group.representative_index]
    crystal_unique_rows.append({
        **representative,
        "canonical": grouping.canonical_structures[group.representative_index],
        "crystal_group_id": "crystal-group-" + str(group_index).zfill(4),
    })
    crystal_group_records.append({
        "group_id": "crystal-group-" + str(group_index).zfill(4),
        "representative_ref": representative["candidate"].candidate_ref.model_dump(mode="json"),
        "member_refs": [
            item["candidate"].candidate_ref.model_dump(mode="json") for item in members
        ],
        "member_profiles": sorted({item["profile_id"] for item in members}),
        "structure_hash": group.representative_hash,
    })

for row in crystal_unique_rows:
    unique_name = (
        row["crystal_group_id"] + "-"
        + row["candidate"].candidate_ref.content_hash[:12] + ".cif"
    )
    unique_path = UNIQUE_CIF_ROOT / unique_name
    unique_path.write_text(row["cif"].value, encoding="utf-8")
    row["unique_cif_path"] = str(unique_path.relative_to(RUN_ROOT))

geometry_valid_rows = []
for row in crystal_unique_rows:
    report = validate_crystal_geometry(
        row["structure"],
        minimum_distance_angstrom=MINIMUM_DISTANCE_A,
        raise_on_error=False,
    )
    if not report.is_valid:
        identity_failures.append({
            "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
            "stage": "geometry-valid",
            "errors": list(report.errors),
        })
        continue
    geometry_valid_rows.append({
        **row,
        "geometry": {
            "atom_count": report.atom_count,
            "volume_angstrom3": report.volume_angstrom3,
            "volume_per_atom_angstrom3": report.volume_per_atom_angstrom3,
            "minimum_distance_angstrom": report.minimum_distance_angstrom,
            "warnings": list(report.warnings),
        },
        "composition_key": str(
            row["canonical"].canonical_structure.composition.reduced_formula
        ),
    })

identity_novelty_assessments = StagedNoveltyAssessor(
    MaterialsProjectStructureLookup.from_environment()
).assess([row["candidate"] for row in geometry_valid_rows])
identity_novelty_by_ref = {
    ref_key(item.candidate_ref): item for item in identity_novelty_assessments
}
external_novelty_status_counts = {status: 0 for status in ("match", "no_match", "unknown")}
for assessment in identity_novelty_assessments:
    external_novelty_status_counts[str(assessment.external_database.status)] += 1

identity_evidence_run = stage_evidence_router.run(
    ValidationEvidenceRequest(
        stage=ValidationEvidenceStage.IDENTITY_NOVELTY,
        chemical_system=chemical_system,
        candidate_refs=[row["candidate"].candidate_ref for row in geometry_valid_rows],
        composition_keys=sorted({row["composition_key"] for row in geometry_valid_rows}),
        observations={
            "exact_file_unique": len(exact_unique_rows),
            "crystallographically_unique": len(crystal_unique_rows),
            "geometry_valid": len(geometry_valid_rows),
            "cross_profile_structure_groups": len(crystal_group_records),
            "materials_project_external_statuses": external_novelty_status_counts,
            "identity_method": "pymatgen.StructureMatcher",
            "absence_is_universal_novelty": False,
        },
        focus=(
            "Retrieve crystallographic reports for the candidate compositions and "
            "possible aliases; database absence must remain scoped or unknown."
        ),
    ),
    goal=profile_inputs[0]["goal"],
)
stage_evidence_runs["identity_novelty"] = identity_evidence_run

print({
    "returned_candidate_cifs_parsed": len(parsed_rows),
    "returned_candidate_exact_unique": len(exact_unique_rows),
    "cross_profile_crystallographically_unique": len(crystal_unique_rows),
    "geometry_valid": len(geometry_valid_rows),
    "cross_profile_groups": len(crystal_group_records),
    "rankable_source_directory": str(UNIQUE_CIF_ROOT),
    "raw_audit_directory_not_ranked": str(AUDIT_CIF_ROOT),
})


In [ ]:
# @title 5. Preserve raw expert evidence, relax twice, and rank by composition
from discovery_os.fusion_schemas import ExpertFeaturePayload, FeatureStatus
from discovery_os.hashing import bytes_hash
from discovery_os.materials_screening import (
    CandidateScreeningVector,
    MLIPScreeningPrediction,
    classify_model_disagreement,
    force_rmse,
    rank_composition_scoped_pareto,
    select_dft_handoff_refs,
)
from discovery_os.sidecars.conversions import periodic_atom_entity_ids
from discovery_os.relaxation import (
    PeriodicRelaxationPayload,
    PeriodicRelaxationRequest,
    PeriodicRelaxationSettings,
)

def properties_by_name(payload: ExpertFeaturePayload) -> dict:
    return {item.property_name: item for item in payload.properties}

def required_prediction(payload: ExpertFeaturePayload) -> MLIPScreeningPrediction:
    properties = properties_by_name(payload)
    energy = properties.get("energy_per_atom")
    maximum_force = properties.get("max_force")
    if energy is None or maximum_force is None:
        raise ValueError(payload.expert_id + " omitted energy_per_atom or max_force.")
    if energy.unit != "eV/atom" or maximum_force.unit != "eV/angstrom":
        raise ValueError(payload.expert_id + " returned unexpected property units.")
    stress = properties.get("stress_norm")
    return MLIPScreeningPrediction(
        expert_id=payload.expert_id,
        energy_per_atom_eV=energy.value,
        max_force_eV_A=maximum_force.value,
        stress_norm=(stress.value if stress is not None else None),
        stress_unit=(stress.unit if stress is not None else None),
    )

def load_verified_feature(reference):
    path = ARTIFACT_ROOT / reference.artifact.relative_path
    resolved = path.resolve()
    resolved.relative_to(ARTIFACT_ROOT.resolve())
    encoded = resolved.read_bytes()
    if len(encoded) != reference.artifact.byte_size:
        raise ValueError("Feature artifact byte size changed.")
    if bytes_hash(encoded) != reference.artifact.sha256:
        raise ValueError("Feature artifact SHA-256 changed.")
    payload = ExpertFeaturePayload.model_validate_json(encoded, strict=True)
    if payload.candidate_ref != reference.candidate_ref:
        raise ValueError("Feature artifact candidate binding changed.")
    if payload.expert_id != reference.expert_id:
        raise ValueError("Feature artifact expert binding changed.")
    return payload

def aligned_force_rows(
    first: ExpertFeaturePayload,
    second: ExpertFeaturePayload,
    candidate: Candidate,
):
    if first.tensor is None or second.tensor is None:
        raise ValueError("Successful MLIP evidence requires force tensors.")
    if first.semantics is None or second.semantics is None:
        raise ValueError("Successful MLIP evidence requires semantics.")
    first_ids = list(first.semantics.entity_ids)
    second_ids = list(second.semantics.entity_ids)
    cif_rows = [
        item for item in candidate.representations
        if item.kind == RepresentationKind.CIF
    ]
    if len(cif_rows) != 1:
        raise ValueError("Force alignment requires one authoritative Candidate CIF.")
    source_structure = parse_crystal_structure(cif_rows[0].value, fmt="cif")
    expected_ids = list(periodic_atom_entity_ids(source_structure))
    if (
        not first_ids
        or len(first_ids) != len(set(first_ids))
        or set(first_ids) != set(second_ids)
        or set(first_ids) != set(expected_ids)
    ):
        raise ValueError(
            "MLIP site identities differ from the authoritative CIF species/coordinates."
        )
    ordered_site_signature = [
        {
            "entity_id": entity_id,
            "species": str(site.species),
            "fractional_coordinates": [
                round(float(value), 12) for value in site.frac_coords
            ],
        }
        for entity_id, site in zip(expected_ids, source_structure, strict=True)
    ]
    def rows(payload):
        columns = payload.tensor.shape[-1]
        if len(payload.tensor.shape) != 2 or columns < 3:
            raise ValueError("Expected an atom-by-feature MLIP tensor.")
        values = payload.tensor.values
        return [
            values[index:index + columns]
            for index in range(0, len(values), columns)
        ]
    first_rows = rows(first)
    second_rows = rows(second)
    second_by_id = dict(zip(second_ids, second_rows, strict=True))
    return (
        first_rows,
        [second_by_id[item] for item in first_ids],
        {
            "basis": "immutable_candidate_ref_plus_species_wrapped_fractional_coordinates",
            "candidate_ref": candidate.candidate_ref.model_dump(mode="json"),
            "ordered_sites": ordered_site_signature,
        },
    )

mlip_rows = []
screening_failures = []
for row in geometry_valid_rows:
    references = {
        reference.expert_id: reference
        for reference in row["after_report"].feature_refs
        if reference.candidate_ref == row["candidate"].candidate_ref
        and reference.expert_id in {"mattersim", "chgnet"}
    }
    if set(references) != {"mattersim", "chgnet"}:
        screening_failures.append({
            "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
            "stage": "mlip-evaluated",
            "error": "both expert references are required",
        })
        continue
    try:
        payloads = {
            expert_id: load_verified_feature(reference)
            for expert_id, reference in references.items()
        }
        if any(
            payload.status != FeatureStatus.SUCCESS
            for payload in payloads.values()
        ):
            raise ValueError("Both MLIP feature calls must have success status.")
        mattersim_prediction = required_prediction(payloads["mattersim"])
        chgnet_prediction = required_prediction(payloads["chgnet"])
        mattersim_forces, chgnet_forces, force_alignment_audit = aligned_force_rows(
            payloads["mattersim"],
            payloads["chgnet"],
            row["candidate"],
        )
        rmse = force_rmse(mattersim_forces, chgnet_forces)
    except Exception as exc:
        screening_failures.append({
            "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
            "stage": "mlip-evaluated",
            "error": type(exc).__name__ + ": " + str(exc),
        })
        continue
    mlip_rows.append({
        **row,
        "expert_payloads": payloads,
        "predictions": {
            "mattersim": mattersim_prediction,
            "chgnet": chgnet_prediction,
        },
        "force_rmse_eV_A": rmse,
        "force_alignment_audit": force_alignment_audit,
        "feature_artifacts": {
            expert_id: references[expert_id].artifact.model_dump(mode="json")
            for expert_id in ("mattersim", "chgnet")
        },
    })

mlip_evidence_run = stage_evidence_router.run(
    ValidationEvidenceRequest(
        stage=ValidationEvidenceStage.MLIP_DISAGREEMENT,
        chemical_system=chemical_system,
        candidate_refs=[row["candidate"].candidate_ref for row in mlip_rows],
        composition_keys=sorted({row["composition_key"] for row in mlip_rows}),
        observations={
            "dual_model_evaluated": len(mlip_rows),
            "models": ["MatterSim", "CHGNet"],
            "energy_unit": "eV/atom",
            "force_unit": "eV/angstrom",
            "maximum_energy_abs_difference_eV_atom": (
                max(
                    abs(
                        row["predictions"]["mattersim"].energy_per_atom_eV
                        - row["predictions"]["chgnet"].energy_per_atom_eV
                    )
                    for row in mlip_rows
                )
                if mlip_rows else None
            ),
            "maximum_force_rmse_eV_A": (
                max(row["force_rmse_eV_A"] for row in mlip_rows)
                if mlip_rows else None
            ),
            "both_stress_outputs_available": sum(
                row["predictions"]["mattersim"].stress_norm is not None
                and row["predictions"]["chgnet"].stress_norm is not None
                for row in mlip_rows
            ),
            "property_scores_created_from_literature": False,
        },
        focus=(
            "Retrieve published applicability limits, magnetic or charge-state "
            "caveats, and calculations that could explain dual-MLIP disagreement."
        ),
    ),
    goal=profile_inputs[0]["goal"],
)
stage_evidence_runs["mlip_disagreement"] = mlip_evidence_run

relaxation_settings = PeriodicRelaxationSettings(
    optimizer="FIRE",
    requested_steps=RELAX_STEPS,
    target_fmax_eV_A=RELAX_FMAX_EV_A,
    relax_cell=True,
    minimum_distance_safety_A=MINIMUM_DISTANCE_A,
    max_abs_volume_change_fraction=0.35,
)
relaxation_rows = []
for row_index, row in enumerate(mlip_rows):
    records = {}
    payloads = {}
    for expert_id, variable in (
        ("mattersim", "MATTERSIM_API_URL"),
        ("chgnet", "CHGNET_API_URL"),
    ):
        request = PeriodicRelaxationRequest(
            candidate=row["candidate"],
            settings=relaxation_settings,
            seed=BASE_SEED + row_index,
        )
        endpoint = os.environ[variable].rstrip("/") + "/v1/relax"
        try:
            response = requests.post(
                endpoint,
                json=request.model_dump(mode="json"),
                timeout=1800,
            )
            response.raise_for_status()
            payload = PeriodicRelaxationPayload.model_validate_json(
                response.content,
                strict=True,
            )
            if payload.candidate_ref != row["candidate"].candidate_ref:
                raise ValueError("Relaxation response cites another candidate.")
            if payload.expert_id != expert_id:
                raise ValueError("Relaxation response cites another expert.")
            payloads[expert_id] = payload
            records[expert_id] = {
                "endpoint_operation": "/v1/relax",
                "request_executed": True,
                "execution_succeeded": payload.execution_succeeded,
                "converged": payload.converged,
                "strict_gate_passed": payload.strict_gate_passed,
                "payload": payload.model_dump(mode="json"),
                "error": None,
            }
        except Exception as exc:
            records[expert_id] = {
                "endpoint_operation": "/v1/relax",
                "request_executed": True,
                "execution_succeeded": False,
                "converged": False,
                "strict_gate_passed": False,
                "payload": None,
                "error": type(exc).__name__ + ": " + str(exc),
            }

    relaxed_structure_match = None
    relaxed_structure_comparison_error = None
    if set(payloads) == {"mattersim", "chgnet"}:
        try:
            relaxed_grouping = group_crystal_structures(
                (
                    payloads["mattersim"].relaxed_structure.value,
                    payloads["chgnet"].relaxed_structure.value,
                )
            )
            relaxed_structure_match = len(relaxed_grouping.groups) == 1
        except Exception as exc:
            relaxed_structure_match = None
            relaxed_structure_comparison_error = (
                type(exc).__name__ + ": " + str(exc)
            )

    disagreement = classify_model_disagreement(
        row["predictions"]["mattersim"],
        row["predictions"]["chgnet"],
        force_rmse_eV_A=row["force_rmse_eV_A"],
        relaxed_structure_match=relaxed_structure_match,
        require_stress_comparison=True,
        require_relaxed_structure_comparison=True,
    )
    both_converged = (
        set(payloads) == {"mattersim", "chgnet"}
        and all(payload.converged and payload.strict_gate_passed for payload in payloads.values())
    )
    relaxation_rows.append({
        **row,
        "relaxation_records": records,
        "relaxation_payloads": payloads,
        "relaxed_structure_match": relaxed_structure_match,
        "relaxed_structure_comparison_error": relaxed_structure_comparison_error,
        "disagreement": disagreement,
        "relaxation_converged": both_converged,
    })

vectors = [
    CandidateScreeningVector(
        candidate_ref=row["candidate"].candidate_ref,
        composition_key=row["composition_key"],
        mattersim=row["predictions"]["mattersim"],
        chgnet=row["predictions"]["chgnet"],
        disagreement=row["disagreement"],
        geometry_valid=True,
        relaxation_gate_passed=row["relaxation_converged"],
    )
    for row in relaxation_rows
]
all_pareto_ranks = rank_composition_scoped_pareto(vectors)
vector_by_ref = {ref_key(item.candidate_ref): item for item in vectors}
rank_by_ref = {ref_key(item.candidate_ref): item for item in all_pareto_ranks}
strict_ranked = [
    item
    for item in all_pareto_ranks
    if vector_by_ref[ref_key(item.candidate_ref)].relaxation_gate_passed
]
relaxation_converged_rows = [
    row for row in relaxation_rows if row["relaxation_converged"]
]

disagreement_risk_counts = {risk: 0 for risk in ("low", "medium", "high")}
for vector in vectors:
    disagreement_risk_counts[str(vector.disagreement.risk)] += 1
relaxation_evidence_run = stage_evidence_router.run(
    ValidationEvidenceRequest(
        stage=ValidationEvidenceStage.RELAXATION_VALIDATION,
        chemical_system=chemical_system,
        candidate_refs=[row["candidate"].candidate_ref for row in relaxation_rows],
        composition_keys=sorted({row["composition_key"] for row in relaxation_rows}),
        observations={
            "independent_relaxers": ["MatterSim", "CHGNet"],
            "requested_attempts": 2 * len(mlip_rows),
            "execution_failures": sum(
                not row["relaxation_records"][expert]["execution_succeeded"]
                for row in relaxation_rows
                for expert in ("mattersim", "chgnet")
            ),
            "strict_gate_failures": sum(
                not row["relaxation_records"][expert]["strict_gate_passed"]
                for row in relaxation_rows
                for expert in ("mattersim", "chgnet")
            ),
            "both_relaxations_converged": len(relaxation_converged_rows),
            "relaxed_structure_mismatches": sum(
                row["relaxed_structure_match"] is False for row in relaxation_rows
            ),
            "relaxed_structure_comparison_unknown": sum(
                row["relaxed_structure_match"] is None for row in relaxation_rows
            ),
            "disagreement_risks": disagreement_risk_counts,
        },
        focus=(
            "Retrieve known phase transformations, mechanical or dynamical "
            "instability, pressure/temperature dependence, and phonon evidence."
        ),
    ),
    goal=profile_inputs[0]["goal"],
)
stage_evidence_runs["relaxation_validation"] = relaxation_evidence_run

print({
    "mlip-evaluated": len(mlip_rows),
    "relaxation-converged": len(relaxation_converged_rows),
    "ranked": len(strict_ranked),
    "high-disagreement-queued": sum(
        item.disagreement.dft_escalation for item in vectors
    ),
})


In [ ]:
# @title 6. Assess novelty, escalate disagreement, and create DFT handoffs
from discovery_os.artifacts import ArtifactStore
from discovery_os.dft_handoff import PortablePeriodicDFTInputBackend
from discovery_os.materials_screening import ThermodynamicValidation
if not relaxation_rows:
    raise RuntimeError("No geometry-valid candidate completed dual-MLIP feature evaluation.")

novelty_assessments = identity_novelty_assessments
novelty_by_ref = identity_novelty_by_ref

selected_dft_refs = select_dft_handoff_refs(
    all_pareto_ranks,
    vectors,
    top_k=DFT_TOP_K,
)
if not selected_dft_refs:
    raise RuntimeError(
        "No DFT handoff candidate passed a relaxation gate or high-disagreement escalation rule."
    )
selected_dft_keys = {ref_key(item) for item in selected_dft_refs}
row_by_ref = {
    ref_key(row["candidate"].candidate_ref): row for row in relaxation_rows
}
selected_dft_candidates = [
    row_by_ref[ref_key(reference)]["candidate"]
    for reference in selected_dft_refs
]
dft_evidence_run = stage_evidence_router.run(
    ValidationEvidenceRequest(
        stage=ValidationEvidenceStage.DFT_HANDOFF,
        chemical_system=chemical_system,
        candidate_refs=selected_dft_refs,
        composition_keys=sorted({
            row_by_ref[ref_key(reference)]["composition_key"]
            for reference in selected_dft_refs
        }),
        observations={
            "shortlist_size": len(selected_dft_refs),
            "top_k_limit": DFT_TOP_K,
            "selection": "composition-scoped Pareto plus high-disagreement escalation",
            "dft_executed": False,
            "required_review": [
                "reference phases",
                "magnetic ordering",
                "functional and Hubbard-U choices",
                "pseudopotentials",
                "cutoff and k-point convergence",
                "phonons",
            ],
        },
        focus=(
            "Retrieve calculation choices and experimental references needed to "
            "review these portable DFT inputs; do not claim a DFT result."
        ),
    ),
    goal=profile_inputs[0]["goal"],
)
stage_evidence_runs["dft_handoff"] = dft_evidence_run
stage_evidence_refs = {
    stage: {
        "report_id": run.report.report_id,
        "stage": str(run.report.stage),
        "status": str(run.report.status),
        "report_relative_path": str(run.report_path.relative_to(RUN_ROOT)),
        "record_count": run.report.record_count,
        "claim_count": run.report.claim_count,
        "branch_count": run.report.branch_count,
        "official_validators": list(run.report.route.official_validators),
        "property_score_created": run.report.property_score_created,
    }
    for stage, run in stage_evidence_runs.items()
}

dft_report = PortablePeriodicDFTInputBackend().prepare_inputs(
    selected_dft_candidates,
    artifact_store=ArtifactStore(ARTIFACT_ROOT),
    top_k=len(selected_dft_candidates),
)
if not 1 <= len(dft_report.packages) <= 5:
    raise RuntimeError("A successful shortlist must create 1-5 DFT input packages.")

dft_package_by_ref = {
    ref_key(package.manifest.candidate_ref): package
    for package in dft_report.packages
}
thermodynamic_nulls = ThermodynamicValidation().model_dump(mode="json")
screening_records = []
for row in relaxation_rows:
    key = ref_key(row["candidate"].candidate_ref)
    rank = rank_by_ref[key]
    novelty = novelty_by_ref[key]
    package = dft_package_by_ref.get(key)
    generator_provenance = profile_reports[
        row["profile_id"]
    ].generation.provenance.model_dump(mode="json")
    screening_records.append({
        "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
        "composition_key": row["composition_key"],
        "authoritative_candidate_cif": {
            "relative_path": row["unique_cif_path"],
            "raw_audit_relative_path": row["raw_audit_cif_path"],
            "exact_file_sha256": row["exact_file_sha256"],
            "crystal_group_id": row["crystal_group_id"],
            "source": "Candidate.representations[kind=cif]",
            "final_export_scope": "cross_profile_crystallographic_representative",
            "raw_audit_cif_counted_or_ranked": False,
            "zip_or_extxyz_merge_performed": False,
        },
        "generation_conditions": {
            "chemical_system": chemical_system,
            "requested_profile": row["conditions"].model_dump(mode="json"),
            "candidate_reported_conditions": row["candidate"].attributes.get(
                "conditions"
            ),
            "generator_provenance": generator_provenance,
            "scientific_role": "generation_target_not_a_measured_property",
        },
        "screening_predictions": {
            "mattersim": row["predictions"]["mattersim"].model_dump(mode="json"),
            "chgnet": row["predictions"]["chgnet"].model_dump(mode="json"),
            "raw_feature_artifacts": row["feature_artifacts"],
            "force_rmse_eV_A": row["force_rmse_eV_A"],
            "force_alignment_audit": row["force_alignment_audit"],
            "model_disagreement": row["disagreement"].model_dump(mode="json"),
            "composition_scoped_pareto": rank.model_dump(mode="json"),
            "entered_final_ranked_funnel": row["relaxation_converged"],
            "cross_stoichiometry_energy_comparison_performed": False,
        },
        "relaxation_validation": {
            "mattersim": row["relaxation_records"]["mattersim"],
            "chgnet": row["relaxation_records"]["chgnet"],
            "both_converged_and_strict_gate_passed": row["relaxation_converged"],
            "relaxed_structure_match": row["relaxed_structure_match"],
            "relaxed_structure_comparison_error": row[
                "relaxed_structure_comparison_error"
            ],
        },
        "novelty": novelty.model_dump(mode="json"),
        "stage_validation_evidence": stage_evidence_refs,
        "dft_escalation": {
            "required_due_to_high_disagreement": row[
                "disagreement"
            ].dft_escalation,
            "queue_status": (
                "packaged"
                if key in selected_dft_keys
                else (
                    "queued_by_disagreement_but_outside_top_k"
                    if row["disagreement"].dft_escalation
                    else "not_selected"
                )
            ),
            "input_package_manifest": (
                package.manifest_artifact.model_dump(mode="json")
                if package is not None else None
            ),
        },
        "thermodynamic_validation": {
            "status": "not_run",
            "dft_executed": False,
            **thermodynamic_nulls,
        },
    })

funnel = {
    "requested_samples": requested_count,
    "raw_model_structures": raw_model_structure_count,
    "parsed_structures": parsed_model_structure_count,
    "exact_file_unique": len(raw_exact_file_hashes),
    "crystallographically_unique": len(crystal_unique_rows),
    "geometry_valid": len(geometry_valid_rows),
    "mlip_evaluated": len(mlip_rows),
    "relaxation_converged": len(relaxation_converged_rows),
    "ranked_candidates": len(strict_ranked),
}
failures = identity_failures + screening_failures
(RUN_ROOT / "funnel.json").write_text(
    json.dumps(funnel, indent=2), encoding="utf-8"
)
(RUN_ROOT / "mattergen-generation-funnels.json").write_text(
    json.dumps({
        "counts": profile_generation_funnels,
        "hashes": profile_generation_funnel_hashes,
        "global_exact_file_sha256s": sorted(raw_exact_file_hashes),
    }, indent=2),
    encoding="utf-8",
)
(RUN_ROOT / "crystal-groups.json").write_text(
    json.dumps(crystal_group_records, indent=2), encoding="utf-8"
)
(RUN_ROOT / "screening-records.json").write_text(
    json.dumps(screening_records, indent=2), encoding="utf-8"
)
(RUN_ROOT / "failures.json").write_text(
    json.dumps(failures, indent=2), encoding="utf-8"
)
(RUN_ROOT / "dft-handoff-report.json").write_text(
    dft_report.model_dump_json(indent=2), encoding="utf-8"
)
(RUN_ROOT / "stage-validation-evidence-index.json").write_text(
    json.dumps({
        "reports": stage_evidence_refs,
        "scientific_role": "search_and_validation_context_only",
        "property_scores_created": False,
    }, indent=2),
    encoding="utf-8",
)
run_summary = {
    "run_id": RUN_ID,
    "chemical_system": chemical_system,
    "profile_allocation": PROFILES,
    "returned_authoritative_candidate_count": returned_candidate_count,
    "raw_audit_cifs_are_not_ranked": True,
    "final_unique_cif_directory": str(UNIQUE_CIF_ROOT.relative_to(RUN_ROOT)),
    "funnel": funnel,
    "dft_package_count": len(dft_report.packages),
    "dft_calculation_executed": dft_report.calculation_executed,
    "scientific_boundary": dft_report.scientific_boundary,
    "stage_validation_evidence": stage_evidence_refs,
    "stage_evidence_property_scores_created": False,
    "credentials_serialized": False,
}
(RUN_ROOT / "run-summary.json").write_text(
    json.dumps(run_summary, indent=2), encoding="utf-8"
)
print(json.dumps(run_summary, indent=2))


## Checks

The final cell enforces the exact funnel names and non-increasing counts, verifies both expert and relaxation records, confirms that no thermodynamic value was invented, and writes a portable ZIP without credentials.

In [ ]:
# @title 7. Enforce scientific boundaries and export the run
EXPECTED_FUNNEL_NAMES = [
    "requested_samples",
    "raw_model_structures",
    "parsed_structures",
    "exact_file_unique",
    "crystallographically_unique",
    "geometry_valid",
    "mlip_evaluated",
    "relaxation_converged",
    "ranked_candidates",
]
if list(funnel) != EXPECTED_FUNNEL_NAMES:
    raise AssertionError("Funnel names or order changed.")
if funnel["requested_samples"] != TOTAL_CANDIDATES:
    raise AssertionError("Requested samples must equal the allocated 8-32 batch.")
if returned_candidate_count != TOTAL_CANDIDATES:
    raise AssertionError("Upstream replacement sampling did not return the requested unique batch.")
if funnel["raw_model_structures"] < funnel["parsed_structures"]:
    raise AssertionError("Raw model structures cannot be below parsed structures.")
if funnel["parsed_structures"] < funnel["exact_file_unique"]:
    raise AssertionError("Parsed raw structures cannot be below global returned exact-file uniques.")
downstream_names = [
    "exact_file_unique", "crystallographically_unique", "geometry_valid",
    "mlip_evaluated", "relaxation_converged", "ranked_candidates",
]
downstream_values = [funnel[name] for name in downstream_names]
if any(left < right for left, right in zip(downstream_values, downstream_values[1:])):
    raise AssertionError("The returned-candidate screening funnel must be non-increasing.")
if len(profile_reports) != 4:
    raise AssertionError("Exactly four fusion-iterate profile reports are required.")
EXPECTED_EVIDENCE_STAGES = {
    "generation_prior",
    "identity_novelty",
    "mlip_disagreement",
    "relaxation_validation",
    "dft_handoff",
}
if set(stage_evidence_runs) != EXPECTED_EVIDENCE_STAGES:
    raise AssertionError("All five stage-specific evidence routes are required.")
for stage, run in stage_evidence_runs.items():
    report = run.report
    if report.property_score_created:
        raise AssertionError(stage + " evidence must not create property scores.")
    if str(report.status) in {"unknown", "skipped"} and not report.reason:
        raise AssertionError(stage + " unknown/skipped evidence requires a reason.")
    if not report.route.official_validators:
        raise AssertionError(stage + " must preserve its official validator route.")
    if report.route.mcp_policy != "configured-tool-only":
        raise AssertionError("Only the administrator-configured MCP tool is allowed.")
for item in profile_inputs:
    context = item["decision_context"]
    if context.evidence_branch_id is not None and not context.evidence_claim_ids:
        raise AssertionError("Evidence-guided generation must stay claim-closed.")
if sum(item["candidate_count"] for item in PROFILES) != TOTAL_CANDIDATES:
    raise AssertionError("Profile allocation changed.")

for record in screening_records:
    if set(record["screening_predictions"]["raw_feature_artifacts"]) != {
        "mattersim", "chgnet"
    }:
        raise AssertionError("Both original feature artifacts must be retained.")
    relaxation = record["relaxation_validation"]
    if set(relaxation).issuperset({"mattersim", "chgnet"}) is False:
        raise AssertionError("Both independent relaxation records are required.")
    if any(
        relaxation[expert]["endpoint_operation"] != "/v1/relax"
        for expert in ("mattersim", "chgnet")
    ):
        raise AssertionError("Relaxations must be direct /v1/relax operations.")
    if record["generation_conditions"]["scientific_role"] != (
        "generation_target_not_a_measured_property"
    ):
        raise AssertionError("Generation conditions crossed the evidence boundary.")
    if record["screening_predictions"][
        "cross_stoichiometry_energy_comparison_performed"
    ]:
        raise AssertionError("Absolute MLIP energies cannot be compared across stoichiometries.")
    thermodynamics = record["thermodynamic_validation"]
    if thermodynamics["status"] != "not_run" or thermodynamics["dft_executed"]:
        raise AssertionError("DFT execution was falsely reported.")
    for field in (
        "formation_energy_eV_atom",
        "computed_energy_above_hull_eV_atom",
        "reference_phase_set",
        "method",
    ):
        if thermodynamics[field] is not None:
            raise AssertionError("Uncalculated thermodynamic fields must remain null.")
    if record["authoritative_candidate_cif"]["raw_audit_cif_counted_or_ranked"]:
        raise AssertionError("Raw audit CIFs must not enter screening or ranking.")
    if record["novelty"]["project_history"]["status"] != "unknown":
        raise AssertionError("Unconfigured project history must remain unknown.")
    predictions = record["screening_predictions"]
    if (
        predictions["mattersim"]["stress_norm"] is not None
        and predictions["chgnet"]["stress_norm"] is not None
        and predictions["model_disagreement"]["stress_norm_abs_diff_GPa"] is None
    ):
        raise AssertionError("Available stress norms must be converted and compared.")
    disagreement = predictions["model_disagreement"]
    if disagreement["risk"] == "high" and not record["dft_escalation"][
        "required_due_to_high_disagreement"
    ]:
        raise AssertionError("High disagreement must enter the DFT escalation queue.")

if dft_report.calculation_executed or not 1 <= len(dft_report.packages) <= 5:
    raise AssertionError("DFT handoff must contain 1-5 non-executed input packages.")
if any(package.manifest.calculation_executed for package in dft_report.packages):
    raise AssertionError("An input package cannot claim a completed DFT calculation.")

for secret_name in (
    "MP_API_KEY",
    "OPENALEX_API_KEY",
    "RAG_MODEL_API_KEY",
    "MATERIAL_RAG_MCP_TOKEN",
):
    secret = os.environ.get(secret_name, "")
    if secret:
        for path in RUN_ROOT.rglob("*"):
            if path.is_file() and secret.encode() in path.read_bytes():
                raise AssertionError(
                    secret_name + " was serialized into " + str(path)
                )
    del secret

archive_path = Path(
    shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT)
)
print(json.dumps({
    "checks": "passed",
    "funnel": funnel,
    "dft_packages": len(dft_report.packages),
    "archive": str(archive_path),
}, indent=2))

if DOWNLOAD_RESULTS_ZIP:
    try:
        from google.colab import files
        files.download(str(archive_path))
    except ImportError:
        print("Not running in Colab; archive remains at", archive_path)


## Next Steps

Review every DFT manifest, choose licensed pseudopotentials, replace all placeholders, and run convergence-tested relax/static calculations outside this notebook. Only then compute reference-consistent formation energies and a phase diagram. Experimental synthesis and characterization remain required before making a discovery claim.